# Drift Detection Analysis

This notebook demonstrates the AEGIS drift detection pipeline using
CUSUM and Wasserstein detectors.

**Goals:**
1. Generate synthetic data with known drift
2. Fit detectors on reference data
3. Detect drift with CUSUM
4. Detect drift with Wasserstein
5. Ensemble detection
6. Visualize drift statistics
7. Summary

In [ ]:
# Cell 1: Imports
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import time

# Project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

import matplotlib.pyplot as plt
print(f"Matplotlib: {plt.matplotlib.__version__}")

In [ ]:
# Cell 2: Generate synthetic drift data
rng = np.random.RandomState(42)

n_reference = 500
n_new = 500
n_features = 5

feature_names = [f"feature_{i}" for i in range(n_features)]

# Reference: standard normal
reference_data = rng.normal(0, 1, size=(n_reference, n_features))

# New data with drift on specific features
new_data = rng.normal(0, 1, size=(n_new, n_features))
new_data[:, 0] += 0.5     # Feature 0: mean shift (+0.5)
new_data[:, 1] *= 1.8     # Feature 1: variance increase
# Feature 2-4: no drift (control)

ref_df = pd.DataFrame(reference_data, columns=feature_names)
new_df = pd.DataFrame(new_data, columns=feature_names)

print(f"Reference data: {ref_df.shape}")
print(f"New data:       {new_df.shape}")
print(f"\nReference statistics:")
print(ref_df.describe().round(3))
print(f"\nNew data statistics:")
print(new_df.describe().round(3))

print(f"\nExpected drift:")
print(f"  Feature 0: mean shift of +0.5")
print(f"  Feature 1: variance increase (x1.8)")
print(f"  Features 2-4: no drift (control)")

In [ ]:
# Cell 3: Fit detectors on reference
from app.ml.drift.cusum_detector import CUSUMDetector
from app.ml.drift.wasserstein_detector import WassersteinDetector

# Create per-feature detectors
cusum_detectors = {}
wasserstein_detectors = {}

for i, fname in enumerate(feature_names):
    # CUSUM detector
    cusum = CUSUMDetector(threshold=5.0, drift_parameter=0.5)
    cusum.fit(reference_data[:, i])
    cusum_detectors[fname] = cusum
    
    # Wasserstein detector
    wass = WassersteinDetector(threshold=0.1, n_permutations=500)
    wass.fit(reference_data[:, i])
    wasserstein_detectors[fname] = wass

print("Detectors fitted on reference data:")
for fname in feature_names:
    c_state = cusum_detectors[fname].get_state()
    w_state = wasserstein_detectors[fname].get_state()
    print(f"  {fname}:")
    print(f"    CUSUM ref_mean={c_state['reference_mean']:.4f}, ref_std={c_state['reference_std']:.4f}")
    print(f"    Wasserstein ref_mean={w_state['reference_mean']:.4f}, ref_std={w_state['reference_std']:.4f}")

In [ ]:
# Cell 4: Detect drift with CUSUM
print("CUSUM Drift Detection Results")
print("=" * 50)

cusum_results = {}
cusum_drift_timeseries = {}

for fname in feature_names:
    i = feature_names.index(fname)
    detector = cusum_detectors[fname]
    
    # Reset before detection
    detector.reset()
    
    # Detect on new data (batch)
    results = detector.detect_batch(new_data[:, i])
    
    # Track CUSUM statistics over time
    stats_over_time = []
    detector2 = CUSUMDetector(threshold=5.0, drift_parameter=0.5)
    detector2.fit(reference_data[:, i])
    for val in new_data[:, i]:
        r = detector2.update(float(val))
        stats_over_time.append(r.statistic)
    cusum_drift_timeseries[fname] = stats_over_time
    
    n_drift = sum(1 for r in results if r.drift_detected)
    max_stat = max(r.statistic for r in results)
    
    cusum_results[fname] = {
        'n_drift_points': n_drift,
        'max_statistic': max_stat,
        'total_points': len(results),
        'drift_ratio': n_drift / max(len(results), 1),
    }
    
    drift_flag = '*** DRIFT ***' if n_drift > 0 else ''
    print(f"  {fname}: drift_points={n_drift}/{len(results)}, "
          f"max_stat={max_stat:.4f} {drift_flag}")

print(f"\nSummary: CUSUM detected drift in "
      f"{sum(1 for r in cusum_results.values() if r['n_drift_points'] > 0)}/{n_features} features")

In [ ]:
# Cell 5: Detect drift with Wasserstein
print("Wasserstein Drift Detection Results")
print("=" * 50)

wasserstein_results = {}

for fname in feature_names:
    i = feature_names.index(fname)
    detector = wasserstein_detectors[fname]
    
    result = detector.detect(new_data[:, i])
    
    wasserstein_results[fname] = {
        'drift_detected': result.drift_detected,
        'status': result.status.value,
        'statistic': result.statistic,
        'mean_estimate': result.mean_estimate,
    }
    
    drift_flag = '*** DRIFT ***' if result.drift_detected else ''
    print(f"  {fname}: W1={result.statistic:.6f}, "
          f"status={result.status.value}, "
          f"mean_est={result.mean_estimate:.4f} {drift_flag}")

n_drift = sum(1 for r in wasserstein_results.values() if r['drift_detected'])
print(f"\nSummary: Wasserstein detected drift in {n_drift}/{n_features} features")

In [ ]:
# Cell 6: Ensemble detection
from app.ml.drift.drift_ensemble import DriftEnsemble

print("Ensemble Drift Detection Results")
print("=" * 50)

ensemble = DriftEnsemble(
    cusum_threshold=5.0,
    wasserstein_threshold=0.1,
    min_reference_samples=10,
)

# Fit on first feature for ensemble-level detection
ensemble.fit(reference_data[:, 0], feature_names=['feature_0'])

# Per-feature ensemble results
ensemble_results = {}
for fname in feature_names:
    i = feature_names.index(fname)
    
    # Create per-feature ensemble
    feat_ensemble = DriftEnsemble(
        cusum_threshold=5.0,
        wasserstein_threshold=0.1,
        min_reference_samples=10,
    )
    feat_ensemble.fit(reference_data[:, i], feature_names=[fname])
    result = feat_ensemble.detect(new_data[:, i], feature_name=fname)
    
    ensemble_results[fname] = result
    
    drift_flag = '*** DRIFT ***' if result.drift_detected else ''
    print(f"  {fname}: drift={result.drift_detected}, "
          f"severity={result.overall_severity}, "
          f"details={result.details} {drift_flag}")

# Summary table
n_ensemble_drift = sum(1 for r in ensemble_results.values() if r.drift_detected)
print(f"\nEnsemble detected drift in {n_ensemble_drift}/{n_features} features")

# Per-feature drift scores
scores = ensemble.get_feature_drift_scores(new_data[:, 0])
print(f"\nFeature drift scores: {scores}")

# Report
report = ensemble.generate_report()
print(f"\nEnsemble report: active_alerts={report['active_alerts']}, "
      f"severity_counts={report['severity_counts']}")

In [ ]:
# Cell 7: Visualize drift statistics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Feature distributions (reference vs new)
for i, fname in enumerate(feature_names):
    ax = axes[0, 0]
    ax.hist(reference_data[:, i], bins=30, alpha=0.5, label=f'{fname} (ref)', density=True)
    ax.hist(new_data[:, i], bins=30, alpha=0.5, label=f'{fname} (new)', density=True)

# Instead, show per-feature comparison for feature 0 and 1
axes[0, 0].hist(reference_data[:, 0], bins=30, alpha=0.6, label='Reference', density=True, color='blue')
axes[0, 0].hist(new_data[:, 0], bins=30, alpha=0.6, label='New (drifted)', density=True, color='red')
axes[0, 0].set_title('Feature 0 Distribution (Mean Shift)')
axes[0, 0].set_xlabel('Value')
axes[0, 0].set_ylabel('Density')
axes[0, 0].legend()

axes[0, 1].hist(reference_data[:, 1], bins=30, alpha=0.6, label='Reference', density=True, color='blue')
axes[0, 1].hist(new_data[:, 1], bins=30, alpha=0.6, label='New (drifted)', density=True, color='red')
axes[0, 1].set_title('Feature 1 Distribution (Variance Change)')
axes[0, 1].set_xlabel('Value')
axes[0, 1].set_ylabel('Density')
axes[0, 1].legend()

# Plot 2: CUSUM statistics over time
for fname in feature_names:
    stats = cusum_drift_timeseries.get(fname, [])
    if stats:
        color = 'red' if cusum_results[fname]['n_drift_points'] > 0 else 'green'
        axes[1, 0].plot(stats, label=fname, alpha=0.8, color=color)

axes[1, 0].axhline(y=5.0, color='black', linestyle='--', label='Threshold', alpha=0.7)
axes[1, 0].set_title('CUSUM Statistics Over Time')
axes[1, 0].set_xlabel('Sample Index')
axes[1, 0].set_ylabel('CUSUM Statistic')
axes[1, 0].legend(fontsize=8)

# Plot 3: Wasserstein distances per feature
w_distances = [wasserstein_results[f]['statistic'] for f in feature_names]
colors = ['red' if wasserstein_results[f]['drift_detected'] else 'green' for f in feature_names]
bars = axes[1, 1].bar(feature_names, w_distances, color=colors, alpha=0.7)
axes[1, 1].axhline(y=0.1, color='black', linestyle='--', label='Threshold')
axes[1, 1].set_title('Wasserstein-1 Distance per Feature')
axes[1, 1].set_xlabel('Feature')
axes[1, 1].set_ylabel('W-1 Distance')
axes[1, 1].legend()

# Add value labels on bars
for bar, dist in zip(bars, w_distances):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                    f'{dist:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Drift Detection Analysis', fontsize=14)
plt.tight_layout()
plt.show()
print("Drift visualizations rendered above.")

In [ ]:
# Cell 8: Summary
print("=" * 60)
print("Drift Detection Analysis - Summary")
print("=" * 60)

print(f"""
Data:
  - Reference: {n_reference} samples, {n_features} features (standard normal)
  - New data: {n_new} samples with known drift:
    - Feature 0: mean shift of +0.5
    - Feature 1: variance increase (x1.8)
    - Features 2-4: no drift

Detection Results:
  CUSUM:
    - Detected drift in {sum(1 for r in cusum_results.values() if r['n_drift_points'] > 0)}/{n_features} features
    - Feature 0: {cusum_results['feature_0']['n_drift_points']} drift points (mean shift detected)
    - Feature 1: {cusum_results['feature_1']['n_drift_points']} drift points

  Wasserstein:
    - Detected drift in {sum(1 for r in wasserstein_results.values() if r['drift_detected'])}/{n_features} features
    - Feature 0: W1={wasserstein_results['feature_0']['statistic']:.6f}
    - Feature 1: W1={wasserstein_results['feature_1']['statistic']:.6f}

  Ensemble (CUSUM OR Wasserstein):
    - Detected drift in {n_ensemble_drift}/{n_features} features

Key Takeaways:
  1. CUSUM is effective for detecting gradual mean shifts.
  2. Wasserstein catches distribution shape changes (variance, tails).
  3. The ensemble (OR strategy) ensures near-zero false negatives.
  4. Control features (2-4) correctly show no drift.
""")